In [1]:
# --------------------------------------------------------------------------------
# 📚 LEARNING RESOURCES
# Quick Start: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/quick_start.md
# Cookbook:    https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# --------------------------------------------------------------------------------

import kaggle_benchmarks as kbench

# --------------------------------------------------------------------------------
# STEP 1: DEFINE YOUR TASK
# The @task decorator turns a standard Python function into a Benchmark task.
# The first parameter must always be `llm` (the model being tested).
# --------------------------------------------------------------------------------
@kbench.task(name="What is Kaggle?", description="Does the LLM know what Kaggle is?")
def what_is_kaggle(llm) -> None:

    # A. Prompt the model
    response: str = llm.prompt("What is Kaggle?")

    # B. Simple Check (Hard Rule)
    # Fast and cheap: Ensure specific keywords exist in the output.
    kbench.assertions.assert_in("platform", response.lower())

    # C. Optional Advanced Check (LLM Judge)
    # Use a helper LLM to evaluate the quality of the answer against criteria.
    assessment = kbench.assertions.assess_response_with_judge(
        response_text=response,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The answer must mention data science or machine learning.",
            "The answer should mention competitions.",
        ]
    )

    # Iterate through the judge's feedback and assert success
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge Criterion '{result.criterion}' should pass: {result.reason}"
        )

# --------------------------------------------------------------------------------
# STEP 2: RUN THE TASK
# We use `kbench.llm` as a placeholder. This allows Kaggle to automatically swap
# in different models later when you use the "Add Models" button in the UI.
# --------------------------------------------------------------------------------
what_is_kaggle.run(kbench.llm)

# Note: To test a specific model locally, you can use the dictionary lookup:
# what_is_kaggle.run(kbench.llms["google/gemini-2.0-flash"])

# --------------------------------------------------------------------------------
# STEP 3: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Try `%autopilot` in a new cell to auto-generate tasks or write your own!
# --------------------------------------------------------------------------------

BokehModel(combine_events=True, render_bundle={'docs_json': {'569c7a39-1020-408d-ba46-ec6a5a190d75': {'version…

In [2]:
# On Kaggle this is already installed. Locally you'd do:
# pip install kaggle-benchmarks

# Let's check the version and available components
try:
    import kaggle_benchmarks as kbench
    print("kaggle_benchmarks loaded successfully")
    print(f"Available top-level attributes: {[x for x in dir(kbench) if not x.startswith('_')]}")
except ImportError:
    print("Not running in Kaggle environment — SDK not available here.")
    print("Navigate to kaggle.com/benchmarks/tasks/new to get a pre-configured notebook.")

kaggle_benchmarks loaded successfully
Available top-level attributes: ['Actor', 'ExecutionMode', 'LLMChat', 'Run', 'Runs', 'Usage', 'actors', 'assertions', 'benchmark', 'chats', 'client', 'clients', 'config', 'content_types', 'contexts', 'envs', 'events', 'judge_llm', 'kaggle', 'llm', 'llms', 'messages', 'orchestration', 'prompting', 'results', 'runs', 'system', 'task', 'tasks', 'tools', 'ui', 'usage', 'user', 'utils']


In [3]:
# This is the canonical pattern from the SDK docs
# (run this inside a Kaggle benchmark task notebook)

import kaggle_benchmarks as kbench

@kbench.task(name="simple_riddle")
def solve_riddle(llm, riddle: str, answer: str):
    """Asks a riddle and checks for a keyword in the answer."""
    response = llm.prompt(riddle)
    kbench.assertions.assert_contains_regex(
        f"(?i){answer}",
        response,
        expectation="LLM should give the right answer."
    )

# Execute the task
solve_riddle.run(
    llm=kbench.llm,
    riddle="What gets wetter as it dries?",
    answer="Towel",
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'b8c8ec88-ca89-40a5-b8e9-a20452f0021c': {'version…

In [4]:
import kaggle_benchmarks as kbench
import json
import re

# A set of questions with varying difficulty
# Good calibration = model scores ~80% on questions it rates 80% confident on
CALIBRATION_QUESTIONS = [
    {"q": "What is the capital of France?", "answer": "Paris", "difficulty": "easy"},
    {"q": "In what year was the Treaty of Westphalia signed?", "answer": "1648", "difficulty": "medium"},
    {"q": "What is the boiling point of tungsten in Celsius?", "answer": "5555", "difficulty": "hard"},
    {"q": "Who wrote the novel 'The Recognitions'?", "answer": "William Gaddis", "difficulty": "hard"},
    {"q": "What does HTTP stand for?", "answer": "HyperText Transfer Protocol", "difficulty": "easy"},
]

METACOG_PROMPT = """Answer the following question. After your answer, rate your confidence 
as a percentage from 0-100 on a new line in this exact format: CONFIDENCE: <number>

Question: {question}"""


@kbench.task(name="metacognition_calibration")
def calibration_task(llm, question: str, answer: str, difficulty: str):
    """
    Tests whether a model's stated confidence is calibrated with its actual accuracy.
    A well-calibrated model should score ~X% on questions it rates X% confidence.
    """
    response = llm.prompt(METACOG_PROMPT.format(question=question))

    # Check if the answer is correct
    answer_correct = answer.lower() in response.lower()

    # Extract stated confidence
    conf_match = re.search(r'CONFIDENCE:\s*(\d+)', response)
    stated_confidence = int(conf_match.group(1)) if conf_match else None

    # We assert two things:
    # 1. The model follows the format (has a confidence rating)
    kbench.assertions.assert_contains_regex(
        r'CONFIDENCE:\s*\d+',
        response,
        expectation="Model should state confidence in the required format."
    )

    # 2. If the model says it's very confident (>90%), it should be right
    # (This is one slice of calibration — a real benchmark would aggregate across many runs)
    if stated_confidence is not None and stated_confidence > 90:
        kbench.assertions.assert_true(
            answer_correct,
            expectation=f"Model rated {stated_confidence}% confidence but got the wrong answer."
        )


# Run across all questions
for item in CALIBRATION_QUESTIONS:
    calibration_task.run(
        llm=kbench.llm,
        question=item["q"],
        answer=item["answer"],
        difficulty=item["difficulty"],
    )

In [5]:
# Quick summary of the competition timeline
import pandas as pd
from datetime import date

timeline = pd.DataFrame({
    'Date': [
        'March 17, 2026',
        'April 16, 2026',
        'April 17 – May 31, 2026',
        'May 22, 2026',
        'June 1, 2026'
    ],
    'Milestone': [
        'Competition Start ← we are here',
        'Final Submission Deadline',
        'Judging Period',
        'Deadline to vote on others\' benchmarks',
        'Results Announced'
    ]
})

print(timeline.to_string(index=False))
print("\nDays remaining to deadline:", (date(2026, 4, 16) - date.today()).days)

                   Date                              Milestone
         March 17, 2026        Competition Start ← we are here
         April 16, 2026              Final Submission Deadline
April 17 – May 31, 2026                         Judging Period
           May 22, 2026 Deadline to vote on others' benchmarks
           June 1, 2026                      Results Announced

Days remaining to deadline: 30


In [6]:
import kaggle_benchmarks as kbench

# Esto nos dará la "anatomía" completa del SDK
help(kbench)

Help on package kaggle_benchmarks:

NAME
    kaggle_benchmarks

DESCRIPTION
    # Copyright 2025 Kaggle Inc.
    #
    # Licensed under the Apache License, Version 2.0 (the "License");
    # you may not use this file except in compliance with the License.
    # You may obtain a copy of the License at
    #
    #     http://www.apache.org/licenses/LICENSE-2.0
    #
    # Unless required by applicable law or agreed to in writing, software
    # distributed under the License is distributed on an "AS IS" BASIS,
    # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
    # See the License for the specific language governing permissions and
    # limitations under the License.

PACKAGE CONTENTS
    _config
    actors (package)
    assertions
    chats
    clients
    content_types (package)
    contexts
    envs (package)
    events
    kaggle (package)
    llm_messages
    messages
    orchestration (package)
    prompting
    results
    runs
    tasks
    tools (pac

In [7]:
import kaggle_benchmarks as kbench

# 1. Listar las tareas disponibles usando el módulo 'tasks'
try:
    # En la v0.2.0, las tareas suelen estar en kbench.tasks.list_tasks() 
    # o simplemente accesibles vía kbench.tasks
    all_tasks = kbench.tasks.list_tasks()
    print(f"✅ ¡Éxito! Encontré {len(all_tasks)} tareas.")
    print(f"Primeras 3 tareas: {all_tasks[:3]}")
    
    # 2. Cargar la primera tarea para ver qué tiene
    first_task_id = all_tasks[0].id
    task_data = kbench.tasks.get_task(first_task_id)
    print(f"\n📝 Descripción de la tarea '{first_task_id}':")
    print(task_data.description)
    
except Exception as e:
    print(f"❌ Error al intentar listar: {e}")
    # Si falla, intentemos ver qué hay dentro de kbench.tasks
    print("\nContenido de kbench.tasks:", [f for f in dir(kbench.tasks) if not f.startswith('_')])

❌ Error al intentar listar: module 'kaggle_benchmarks.tasks' has no attribute 'list_tasks'

Contenido de kbench.tasks: ['Any', 'Callable', 'Generic', 'Iterable', 'NonRecoverableError', 'Self', 'T', 'TYPE_CHECKING', 'Task', 'TypeVar', 'benchmark', 'chats', 'dataclasses', 'datetime', 'events', 'functools', 'inspect', 'logger', 'logging', 'pd', 'results', 'task', 'time', 'utils']


In [8]:
import os

print("🔍 Buscando carpetas de datos conectadas...")
# Listar todo lo que hay en /kaggle/input para ver el nombre exacto
if os.path.exists('/kaggle/input'):
    datasets = os.listdir('/kaggle/input')
    print(f"Directorios encontrados: {datasets}")
    
    # Si hay carpetas, vamos a ver qué hay dentro de la primera
    for dataset in datasets:
        path = f'/kaggle/input/{dataset}'
        print(f"\nContenido de {path}:")
        try:
            print(os.listdir(path))
        except:
            print("No se pudo leer esta carpeta.")
else:
    print("❌ No hay nada en /kaggle/input. ¿Le diste al botón '+ Add Data' o aceptaste las reglas de la competencia?")

🔍 Buscando carpetas de datos conectadas...
Directorios encontrados: ['competitions']

Contenido de /kaggle/input/competitions:
['kaggle-measuring-agi']


In [9]:
import os
ruta_base = '/kaggle/input/competitions/kaggle-measuring-agi'
print(f"📦 Archivos dentro de la competencia:")
print(os.listdir(ruta_base))

📦 Archivos dentro de la competencia:
['NOTE.md']


In [10]:
ruta_nota = '/kaggle/input/competitions/kaggle-measuring-agi/NOTE.md'

with open(ruta_nota, 'r') as f:
    print("📜 CONTENIDO DE NOTE.md:\n")
    print(f.read())

📜 CONTENIDO DE NOTE.md:

Welcome!

This is a Hackathon with no provided dataset.

Please refer to kaggle.com/competitions/kaggle-measuring-agi/data for data inspiration.


In [11]:
import kaggle_benchmarks as kbench

@kbench.tasks.task(name="Logistica de Almacen Abstracta")
def tarea_humberto(input_data):
    """
    REGLA: Hay una cuadrícula (almacén). El número 2 es una caja. 
    El número 1 es un obstáculo. El 3 es el muelle de carga.
    La IA debe devolver la ruta más corta (lista de coordenadas).
    """
    # Aquí es donde la IA pondría su respuesta
    # Por ahora, definimos la estructura
    return "Ruta Optimizada"

# Para probar si el SDK la reconoce:
print("Tarea registrada correctamente en el sistema de Kaggle.")

Tarea registrada correctamente en el sistema de Kaggle.


In [12]:
import numpy as np
import random

def generar_mapa_almacen(filas=10, cols=10):
    # 0 = Pasillo libre, 1 = Estantería (obstáculo)
    mapa = np.zeros((filas, cols), dtype=int)
    
    # Crear pasillos tipo estantería (columnas impares son estantes)
    for j in range(1, cols, 2):
        for i in range(1, filas - 1):
            mapa[i, j] = 1
            
    # Definir punto de inicio (Entrada al CEDI)
    inicio = (0, 0)
    
    # Definir 3 productos a recoger (Picking List)
    productos = []
    while len(productos) < 3:
        pos = (random.randint(0, filas-1), random.randint(0, cols-1))
        if mapa[pos] == 0 and pos != inicio:
            productos.append(pos)
            
    return mapa, inicio, productos

# Visualicemos el reto
mapa, inicio, lista_picking = generar_mapa_almacen()
print("🗺️ LAYOUT DEL ALMACÉN (1=Estante, 0=Pasillo):")
print(mapa)
print(f"\n📍 Inicio en: {inicio}")
print(f"📦 Lista de Picking (debes recoger estos 3): {lista_picking}")

🗺️ LAYOUT DEL ALMACÉN (1=Estante, 0=Pasillo):
[[0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 0 0 0 0 0 0 0 0 0]]

📍 Inicio en: (0, 0)
📦 Lista de Picking (debes recoger estos 3): [(9, 6), (1, 6), (3, 2)]


In [13]:
def añadir_montacargas(mapa, lista_picking):
    # 4 = Montacargas (Bloqueo temporal en un pasillo)
    # Buscamos un pasillo libre (0) que no sea donde están los productos
    intentos = 0
    while intentos < 20:
        f = random.randint(0, 9)
        c = random.randint(0, 9)
        if mapa[f, c] == 0 and (f, c) not in lista_picking and (f, c) != (0,0):
            mapa[f, c] = 4
            return (f, c)
        intentos += 1
    return None

# Actualizamos tu mapa actual
pos_montacargas = añadir_montacargas(mapa, lista_picking)
print("🏗️ NUEVO MAPA CON MONTACARGAS (4):")
print(mapa)
print(f"\n⚠️ ALERTA: Montacargas bloqueando en {pos_montacargas}")

🏗️ NUEVO MAPA CON MONTACARGAS (4):
[[0 0 0 0 0 0 0 0 0 0]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 4 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 0 0 0 0 0 0 0 0 0]]

⚠️ ALERTA: Montacargas bloqueando en (2, 6)


In [14]:
def añadir_bloqueo_critico(mapa, lista_picking):
    # Elegimos una cabecera (Fila 0 o 9) y una columna de pasillo (Pares: 2, 4, 6, 8)
    filas_criticas = [0, 9]
    columnas_pasillo = [2, 4, 6, 8]
    
    intentos = 0
    while intentos < 10:
        f = random.choice(filas_criticas)
        c = random.choice(columnas_pasillo)
        
        # Verificamos que no estemos bloqueando exactamente donde está un producto
        if (f, c) not in lista_picking:
            mapa[f, c] = 4
            return (f, c)
        intentos += 1
    return None

# Aplicamos el bloqueo de cruce
pos_critica = añadir_bloqueo_critico(mapa, lista_picking)
print("🏗️ LAYOUT CON BLOQUEO ESTRATÉGICO (4):")
print(mapa)
print(f"\n🚫 CRUCE BLOQUEADO EN: {pos_critica}")

🏗️ LAYOUT CON BLOQUEO ESTRATÉGICO (4):
[[0 0 0 0 0 0 0 0 4 0]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 4 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 1 0 1 0 1 0 1 0 1]
 [0 0 0 0 0 0 0 0 0 0]]

🚫 CRUCE BLOQUEADO EN: (0, 8)


In [15]:
import json
import numpy as np
import random

def generar_caso_completo(id_caso):
    # Crear layout 10x10
    mapa = np.zeros((10, 10), dtype=int)
    for j in range(1, 10, 2):
        for i in range(1, 9):
            mapa[i, j] = 1
            
    # Lista de picking (3 productos en pasillos)
    productos = []
    while len(productos) < 3:
        pos = (random.randint(0, 9), random.choice([0, 2, 4, 6, 8]))
        if pos != (0, 0) and pos not in productos:
            productos.append(pos)
            
    # Bloqueo Crítico (Fila 0 o 9 en un cruce de pasillos)
    f_bloqueo = random.choice([0, 9])
    c_bloqueo = random.choice([2, 4, 6, 8])
    mapa[f_bloqueo, c_bloqueo] = 4
    
    return {
        "case_id": id_caso,
        "map": mapa.tolist(),
        "start": [0, 0],
        "picking_list": [list(p) for p in productos],
        "block_point": [f_bloqueo, c_bloqueo],
        "instructions": "Encuentra la ruta más corta desde (0,0) recogiendo los 3 productos y evitando estantes (1) y bloqueos (4)."
    }

# Generar 5 casos
benchmark_data = [generar_caso_completo(i) for i in range(5)]

# Guardar en archivo JSON
with open('humberto_almacen_benchmark.json', 'w') as f:
    json.dump(benchmark_data, f, indent=4)

print("✅ Archivo 'humberto_almacen_benchmark.json' creado con éxito.")

✅ Archivo 'humberto_almacen_benchmark.json' creado con éxito.
